<h2>SQL QUERY</h2>

<h3>Load Datasets& Data Cleaning</h3>

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import requests
import time

In [2]:
def load_socrata_dataset(base_url, params, limit=25000, sleep_time=0.2):
    all_rows = []
    offset = 0
    page = 1

    while True:
        page_params = params.copy()
        page_params["$limit"] = limit
        page_params["$offset"] = offset

        print(f"Requesting page {page}, offset {offset}...")

        response = requests.get(base_url, params=page_params, timeout=120)
        response.raise_for_status()

        rows = response.json()

        if len(rows) == 0:
            print("Download complete.")
            break

        all_rows.extend(rows)
        print(f"Downloaded {len(all_rows):,} rows so far.")

        offset += limit
        page += 1
        time.sleep(sleep_time)

    return pd.DataFrame(all_rows)

In [3]:
crime_url = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"

crime_params = {
    "$select": (
        "id, date, primary_type, description, location_description, "
        "arrest, domestic, beat, district, ward, community_area, latitude, longitude"
    ),
    "$where": "date >= '2020-01-01T00:00:00' AND date < '2025-01-01T00:00:00'",
    "$order": "date, id"
}

crime = load_socrata_dataset(crime_url, crime_params)

# Remove exact duplicate rows that may appear during API pagination
crime = crime.drop_duplicates().copy()

print("Crime dataset shape:", crime.shape)

Requesting page 1, offset 0...
Downloaded 25,000 rows so far.
Requesting page 2, offset 25000...
Downloaded 50,000 rows so far.
Requesting page 3, offset 50000...
Downloaded 75,000 rows so far.
Requesting page 4, offset 75000...
Downloaded 100,000 rows so far.
Requesting page 5, offset 100000...
Downloaded 125,000 rows so far.
Requesting page 6, offset 125000...
Downloaded 150,000 rows so far.
Requesting page 7, offset 150000...
Downloaded 175,000 rows so far.
Requesting page 8, offset 175000...
Downloaded 200,000 rows so far.
Requesting page 9, offset 200000...
Downloaded 225,000 rows so far.
Requesting page 10, offset 225000...
Downloaded 250,000 rows so far.
Requesting page 11, offset 250000...
Downloaded 275,000 rows so far.
Requesting page 12, offset 275000...
Downloaded 300,000 rows so far.
Requesting page 13, offset 300000...
Downloaded 325,000 rows so far.
Requesting page 14, offset 325000...
Downloaded 350,000 rows so far.
Requesting page 15, offset 350000...
Downloaded 375,00

In [4]:
socio_url = "https://data.cityofchicago.org/resource/kn9c-c2s2.json"

socio_params = {
    "$select": (
        "ca, community_area_name, "
        "percent_of_housing_crowded, "
        "percent_households_below_poverty, "
        "percent_aged_16_unemployed, "
        "percent_aged_25_without_high_school_diploma, "
        "percent_aged_under_18_or_over_64, "
        "per_capita_income_, hardship_index"
    )
}

socio_raw = load_socrata_dataset(socio_url, socio_params, limit=5000)

socio = socio_raw.rename(columns={
    "ca": "community_area",
    "percent_of_housing_crowded": "crowded_housing_rate",
    "percent_households_below_poverty": "poverty_rate",
    "percent_aged_16_unemployed": "unemployment_rate",
    "percent_aged_25_without_high_school_diploma": "no_high_school_diploma_rate",
    "percent_aged_under_18_or_over_64": "dependency_rate",
    "per_capita_income_": "per_capita_income"
})

print("Socioeconomic dataset shape:", socio.shape)
socio.head()

Requesting page 1, offset 0...
Downloaded 78 rows so far.
Requesting page 2, offset 5000...
Download complete.
Socioeconomic dataset shape: (78, 9)


,community_area,community_area_name,crowded_housing_rate,poverty_rate,unemployment_rate,no_high_school_diploma_rate,dependency_rate,per_capita_income,hardship_index
0,1,Rogers Park,7.7,23.6,8.7,18.2,27.5,23939,39
1,2,West Ridge,7.8,17.2,8.8,20.8,38.5,23040,46
2,3,Uptown,3.8,24,8.9,11.8,22.2,35787,20
3,4,Lincoln Square,3.4,10.9,8.2,13.4,25.5,37524,17
4,5,North Center,0.3,7.5,5.2,4.5,26.2,57123,6


In [5]:
crime_clean = crime.copy()

crime_columns = [
    "id", "date", "primary_type", "description", "location_description",
    "arrest", "domestic", "beat", "district", "ward",
    "community_area", "latitude", "longitude"
]

for col in crime_columns:
    if col not in crime_clean.columns:
        crime_clean[col] = np.nan

crime_clean = crime_clean[crime_columns]

# Convert date
crime_clean["date"] = pd.to_datetime(crime_clean["date"], errors="coerce")

# Convert numeric columns
numeric_crime_columns = [
    "id", "beat", "district", "ward",
    "community_area", "latitude", "longitude"
]

for col in numeric_crime_columns:
    crime_clean[col] = pd.to_numeric(crime_clean[col], errors="coerce")

# Convert arrest and domestic to 0/1
crime_clean["arrest"] = crime_clean["arrest"].astype(str).str.lower().map({
    "true": 1,
    "false": 0,
    "1": 1,
    "0": 0
})

crime_clean["domestic"] = crime_clean["domestic"].astype(str).str.lower().map({
    "true": 1,
    "false": 0,
    "1": 1,
    "0": 0
})

# Remove records without community_area
crime_clean = crime_clean.dropna(subset=["community_area"]).copy()
crime_clean["community_area"] = crime_clean["community_area"].astype(int)

# Create time-based features
crime_clean["year"] = crime_clean["date"].dt.year
crime_clean["month"] = crime_clean["date"].dt.month
crime_clean["weekday"] = crime_clean["date"].dt.day_name()
crime_clean["hour"] = crime_clean["date"].dt.hour

crime_clean["is_weekend"] = crime_clean["weekday"].isin(
    ["Saturday", "Sunday"]
).astype(int)

# Create violent crime indicator
violent_crimes = [
    "HOMICIDE",
    "CRIMINAL SEXUAL ASSAULT",
    "CRIM SEXUAL ASSAULT",
    "ROBBERY",
    "ASSAULT",
    "BATTERY",
    "KIDNAPPING"
]

crime_clean["is_violent"] = crime_clean["primary_type"].isin(
    violent_crimes
).astype(int)

print("crime_clean shape:", crime_clean.shape)

crime_clean shape: (1184894, 19)


In [6]:
socio_clean = socio.copy()

socio_columns = [
    "community_area",
    "community_area_name",
    "crowded_housing_rate",
    "poverty_rate",
    "unemployment_rate",
    "no_high_school_diploma_rate",
    "dependency_rate",
    "per_capita_income",
    "hardship_index"
]

for col in socio_columns:
    if col not in socio_clean.columns:
        socio_clean[col] = np.nan

socio_clean = socio_clean[socio_columns]

# Convert numeric columns
numeric_socio_columns = [
    "community_area",
    "crowded_housing_rate",
    "poverty_rate",
    "unemployment_rate",
    "no_high_school_diploma_rate",
    "dependency_rate",
    "per_capita_income",
    "hardship_index"
]

for col in numeric_socio_columns:
    socio_clean[col] = pd.to_numeric(socio_clean[col], errors="coerce")

# Keep only Chicago's 77 community areas
# The API includes one citywide summary row, which is not a community area
socio_clean = socio_clean[
    socio_clean["community_area"].between(1, 77)
].copy()

socio_clean["community_area"] = socio_clean["community_area"].astype(int)

print("socio_clean shape:", socio_clean.shape)
socio_clean.head()

socio_clean shape: (77, 9)


,community_area,community_area_name,crowded_housing_rate,poverty_rate,unemployment_rate,no_high_school_diploma_rate,dependency_rate,per_capita_income,hardship_index
0,1,Rogers Park,7.7,23.6,8.7,18.2,27.5,23939,39.0
1,2,West Ridge,7.8,17.2,8.8,20.8,38.5,23040,46.0
2,3,Uptown,3.8,24.0,8.9,11.8,22.2,35787,20.0
3,4,Lincoln Square,3.4,10.9,8.2,13.4,25.5,37524,17.0
4,5,North Center,0.3,7.5,5.2,4.5,26.2,57123,6.0


In [7]:
conn = sqlite3.connect("chicago_crime_project.db")

crime_clean.to_sql("crime", conn, if_exists="replace", index=False)
socio_clean.to_sql("socio", conn, if_exists="replace", index=False)

def run_sql(query):
    return pd.read_sql_query(query, conn)

print("SQLite tables created: crime, socio")

SQLite tables created: crime, socio


In [8]:
conn.execute("DROP TABLE IF EXISTS crime_area_summary;")

conn.execute("""
CREATE TABLE crime_area_summary AS
SELECT
    community_area,
    COUNT(*) AS total_crimes,
    SUM(is_violent) AS violent_crimes,
    1.0 * SUM(is_violent) / COUNT(*) AS violent_crime_share,
    AVG(arrest) AS arrest_rate,
    AVG(domestic) AS domestic_share
FROM crime
GROUP BY community_area;
""")

conn.commit()

print("SQLite summary table created: crime_area_summary")

SQLite summary table created: crime_area_summary


<h3> Part A: SQL Data Validation and Analysis Setup (Query 1–3)

<h3>Query 1: Crime Table Size Validation</h3>

In [16]:
run_sql('''
SELECT
    COUNT(*) AS total_crime_records
FROM crime;
''')

,total_crime_records
0,1184894


This query checks the number of cleaned incident-level crime records loaded into the SQLite `crime` table. It is a data validation query rather than a substantive finding. Before using SQL for aggregation, joins, and community-level comparisons, I first confirm that the SQL database contains the expected cleaned crime dataset.

The output shows 1,184,894 total crime records. This confirms that the incident-level data were successfully loaded into SQLite and provides the baseline record count for the rest of the SQL analysis. Since later queries aggregate these records to the community-area level, this check helps ensure that the SQL results are built from the full cleaned dataset rather than an incomplete table.

<h3>Query 2: Community Area Coverage Validation

In [17]:
run_sql("""
SELECT
    (SELECT COUNT(DISTINCT community_area) FROM crime) AS communities_in_crime_data,
    (SELECT COUNT(DISTINCT community_area) FROM socio) AS communities_in_socio_data,
    (SELECT COUNT(DISTINCT community_area) FROM crime_area_summary) AS communities_in_summary;
""")

,communities_in_crime_data,communities_in_socio_data,communities_in_summary
0,77,77,77


This query checks community-area coverage across the three SQL tables used in the analysis: the incident-level `crime` table, the socioeconomic `socio` table, and the aggregated `crime_area_summary` table. Since the Python analysis uses Chicago’s 77 community areas as the final unit of analysis, all three tables should contain the same 77 community areas before later joins and neighborhood-level comparisons are performed.

The output shows that all three tables contain 77 distinct community areas. This confirms that the SQL database preserves the same geographic scope as the Python notebook and that later community-level comparisons are based on complete citywide coverage. This step is important because missing community areas in either the crime or socioeconomic table could bias the later analysis.

<h3>Query 3: Joined Community-Level Summary Statistics</h3>

In [18]:
run_sql('''
SELECT
    COUNT(*) AS number_of_community_areas,

    -- Crime-side summary
    ROUND(AVG(c.total_crimes), 1) AS avg_total_crimes,
    MIN(c.total_crimes) AS min_total_crimes,
    MAX(c.total_crimes) AS max_total_crimes,
    ROUND(AVG(c.violent_crime_share), 4) AS avg_violent_crime_share,
    ROUND(AVG(c.domestic_share), 4) AS avg_domestic_share,
    ROUND(AVG(c.arrest_rate), 4) AS avg_arrest_rate,

    -- Socioeconomic-side summary
    ROUND(AVG(s.poverty_rate), 1) AS avg_poverty_rate,
    ROUND(AVG(s.unemployment_rate), 1) AS avg_unemployment_rate,
    ROUND(AVG(s.no_high_school_diploma_rate), 1) AS avg_no_high_school_diploma_rate,
    ROUND(AVG(s.crowded_housing_rate), 1) AS avg_crowded_housing_rate,
    ROUND(AVG(s.dependency_rate), 1) AS avg_dependency_rate,
    ROUND(AVG(s.per_capita_income), 0) AS avg_per_capita_income,
    ROUND(AVG(s.hardship_index), 1) AS avg_hardship_index

FROM crime_area_summary c
JOIN socio s
    ON c.community_area = s.community_area;
''')

,number_of_community_areas,avg_total_crimes,min_total_crimes,max_total_crimes,avg_violent_crime_share,avg_domestic_share,avg_arrest_rate,avg_poverty_rate,avg_unemployment_rate,avg_no_high_school_diploma_rate,avg_crowded_housing_rate,avg_dependency_rate,avg_per_capita_income,avg_hardship_index
0,77,15388.2,1396,62510,0.3057,0.1957,0.1185,21.8,15.4,20.3,4.9,35.7,25563.0,49.5


This query summarizes the joined community-level crime and socioeconomic data. It joins the aggregated `crime_area_summary` table with the `socio` table, so it checks that the main crime measures and socioeconomic indicators can be analyzed together at the community-area level.

The output shows 77 community areas, confirming that the joined summary still covers the full Chicago community-area structure. On the crime side, the average community has about 15,388 reported crimes, but the range is wide: from 1,396 to 62,510 reported crimes. This large difference supports the later analysis of crime concentration across neighborhoods. The average violent-crime share is 0.3057, meaning roughly 30.6% of reported incidents are classified as violent, while the average domestic-related share is 0.1957 and the average arrest rate is 0.1185.

On the socioeconomic side, the joined data include broader indicators beyond poverty and hardship. The average poverty rate is 21.8%, average unemployment rate is 15.4%, average no-high-school-diploma rate is 20.3%, average crowded-housing rate is 4.9%, average dependency rate is 35.7%, and average per capita income is about $25,563. These baseline statistics confirm that the SQL analysis can compare crime profiles with a broader set of socioeconomic indicators in later queries.

<h2> Part B: SQL Drill-Down on Python EDA Findings

<h3> Query 4: High-Crime Communities by Socioeconomic Type

In [19]:
run_sql('''
WITH top_crime_communities AS (
    SELECT
        s.community_area_name,
        c.total_crimes,
        c.violent_crimes,
        c.violent_crime_share,
        c.domestic_share,
        s.hardship_index,
        s.poverty_rate,
        s.per_capita_income,
        s.unemployment_rate,
        s.no_high_school_diploma_rate,
        s.crowded_housing_rate,
        CASE
            WHEN s.hardship_index >= (SELECT AVG(hardship_index) FROM socio)
                THEN 'High-volume / higher-hardship'
            ELSE 'High-volume / lower-hardship'
        END AS high_crime_type
    FROM crime_area_summary c
    JOIN socio s
        ON c.community_area = s.community_area
    ORDER BY c.total_crimes DESC
    LIMIT 10
)

SELECT
    high_crime_type,
    COUNT(*) AS num_communities,
    ROUND(AVG(total_crimes), 0) AS avg_total_crimes,
    ROUND(AVG(violent_crime_share) * 100, 1) AS avg_violent_share_pct,
    ROUND(AVG(domestic_share) * 100, 1) AS avg_domestic_share_pct,
    ROUND(AVG(hardship_index), 1) AS avg_hardship_index,
    ROUND(AVG(poverty_rate), 1) AS avg_poverty_rate,
    ROUND(AVG(unemployment_rate), 1) AS avg_unemployment_rate,
    ROUND(AVG(no_high_school_diploma_rate), 1) AS avg_no_high_school_diploma_rate,
    ROUND(AVG(crowded_housing_rate), 1) AS avg_crowded_housing_rate,
    ROUND(AVG(per_capita_income), 0) AS avg_per_capita_income,
    GROUP_CONCAT(community_area_name, ', ') AS example_communities
FROM top_crime_communities
GROUP BY high_crime_type
ORDER BY avg_hardship_index DESC;
''')

,high_crime_type,num_communities,avg_total_crimes,avg_violent_share_pct,avg_domestic_share_pct,avg_hardship_index,avg_poverty_rate,avg_unemployment_rate,avg_no_high_school_diploma_rate,avg_crowded_housing_rate,avg_per_capita_income,example_communities
0,High-volume / higher-hardship,6,39004.0,37.0,26.6,73.3,32.3,22.1,22.7,6.5,15664.0,"Austin, South Shore, North Lawndale, Humboldt ..."
1,High-volume / lower-hardship,4,41153.0,24.5,8.0,7.3,15.7,7.5,7.0,2.4,60521.0,"Near North Side, Near West Side, Loop, West Town"


This query extends the Python crime-concentration finding by examining the top 10 highest-crime communities in more detail. Instead of only showing which communities have the most reported crimes, the SQL query classifies them into two groups based on whether their hardship index is above or below the citywide average.

The output shows that high-volume communities are not all the same. Among the top 10 highest-crime communities, 6 are classified as high-volume / higher-hardship and 4 are classified as high-volume / lower-hardship. Although the two groups have similar average total crime volume, their crime composition is very different. The higher-hardship group has an average violent-crime share of 37.0% and domestic-related share of 26.6%, compared with 24.5% and 8.0% for the lower-hardship group.

The socioeconomic differences are also substantial. The higher-hardship group has higher poverty, unemployment, educational disadvantage, and crowded housing, while the lower-hardship group has much higher average per capita income. This suggests that high crime volume can reflect different underlying neighborhood conditions. Some high-volume communities appear connected to broader socioeconomic disadvantage, while others may reflect commercial activity, visitor exposure, or reporting density.

This query makes the SQL analysis complementary to the Python notebook. Python identifies the broad concentration pattern, while SQL shows that the high-crime communities behind that pattern fall into different socioeconomic types.

<h2> Query 5: High-Crime Communities — Higher-Hardship vs Lower-Hardship Comparison

In [20]:
run_sql('''
WITH high_crime_communities AS (
    SELECT
        s.community_area_name,
        c.total_crimes,
        c.violent_crime_share,
        c.domestic_share,
        c.arrest_rate,
        s.hardship_index,
        s.poverty_rate,
        s.unemployment_rate,
        s.no_high_school_diploma_rate,
        s.crowded_housing_rate,
        s.dependency_rate,
        s.per_capita_income,
        CASE
            WHEN s.hardship_index >= (SELECT AVG(hardship_index) FROM socio)
                THEN 'High-crime / higher-hardship'
            ELSE 'High-crime / lower-hardship'
        END AS high_crime_group
    FROM crime_area_summary c
    JOIN socio s
        ON c.community_area = s.community_area
    WHERE c.total_crimes > (
        SELECT AVG(total_crimes)
        FROM crime_area_summary
    )
)

SELECT
    high_crime_group,
    COUNT(*) AS num_communities,
    ROUND(AVG(total_crimes), 0) AS avg_total_crimes,
    ROUND(AVG(violent_crime_share) * 100, 1) AS avg_violent_share_pct,
    ROUND(AVG(domestic_share) * 100, 1) AS avg_domestic_share_pct,
    ROUND(AVG(arrest_rate) * 100, 1) AS avg_arrest_rate_pct,
    ROUND(AVG(hardship_index), 1) AS avg_hardship_index,
    ROUND(AVG(poverty_rate), 1) AS avg_poverty_rate,
    ROUND(AVG(unemployment_rate), 1) AS avg_unemployment_rate,
    ROUND(AVG(no_high_school_diploma_rate), 1) AS avg_no_high_school_diploma_rate,
    ROUND(AVG(crowded_housing_rate), 1) AS avg_crowded_housing_rate,
    ROUND(AVG(dependency_rate), 1) AS avg_dependency_rate,
    ROUND(AVG(per_capita_income), 0) AS avg_per_capita_income,
    GROUP_CONCAT(community_area_name, ', ') AS example_communities
FROM high_crime_communities
GROUP BY high_crime_group
ORDER BY avg_hardship_index DESC;
''')

,high_crime_group,num_communities,avg_total_crimes,avg_violent_share_pct,avg_domestic_share_pct,avg_arrest_rate_pct,avg_hardship_index,avg_poverty_rate,avg_unemployment_rate,avg_no_high_school_diploma_rate,avg_crowded_housing_rate,avg_dependency_rate,avg_per_capita_income,example_communities
0,High-crime / higher-hardship,20,26984.0,36.6,25.8,15.8,75.0,31.4,22.2,25.6,6.5,39.9,15253.0,"Belmont Cragin, Humboldt park, Austin, West Ga..."
1,High-crime / lower-hardship,10,28976.0,25.1,10.0,10.9,16.4,16.8,7.4,10.0,3.4,23.3,48837.0,"Rogers Park, West Ridge, Uptown, Lake View, Li..."


This query builds on Query 4 by expanding the comparison from the top 10 highest-crime communities to all communities with above-average total crime volume. The purpose is to test whether the same distinction between higher-hardship and lower-hardship high-crime communities still appears when the definition of “high crime” is broader.

The output shows that among above-average crime communities, 20 fall into the high-crime / higher-hardship group and 10 fall into the high-crime / lower-hardship group. The two groups have similar average total crime volume, with about 26,984 crimes in the higher-hardship group and 28,976 crimes in the lower-hardship group. However, their crime composition and socioeconomic conditions are very different. The higher-hardship group has a much higher average violent-crime share, 36.6% compared with 25.1%, and a much higher domestic-related share, 25.8% compared with 10.0%. It also has a higher arrest rate, 15.8% compared with 10.9%.

The socioeconomic contrast is also clear. The high-crime / higher-hardship group has an average hardship index of 75.0, poverty rate of 31.4%, unemployment rate of 22.2%, and substantially higher educational disadvantage. In contrast, the high-crime / lower-hardship group has an average hardship index of only 16.4, poverty rate of 16.8%, and unemployment rate of 7.4%.

This result is important because the two groups have similar crime volume, but very different violent-crime and domestic-related shares. That directly supports the project’s main argument: total reported crime count alone is not enough. SQL adds depth to the Python analysis by showing that among high-crime communities, socioeconomic context helps distinguish between different types of public safety challenges.

<h3> Query 6: Crime Profile and Broader Socioeconomic Conditions by Hardship Group

In [21]:
run_sql('''
SELECT
    CASE
        WHEN s.hardship_index < 50 THEN 'Low Hardship'
        WHEN s.hardship_index >= 50 AND s.hardship_index < 74 THEN 'Mid Hardship'
        WHEN s.hardship_index >= 74 THEN 'High Hardship'
    END AS hardship_group,

    COUNT(*) AS num_communities,

    -- Crime profile
    ROUND(AVG(c.total_crimes), 0) AS avg_total_crimes,
    ROUND(AVG(c.violent_crimes), 0) AS avg_violent_crimes,
    ROUND(AVG(c.violent_crime_share) * 100, 1) AS avg_violent_share_pct,
    ROUND(AVG(c.domestic_share) * 100, 1) AS avg_domestic_share_pct,
    ROUND(AVG(c.arrest_rate) * 100, 1) AS avg_arrest_rate_pct,

    -- Broader socioeconomic profile
    ROUND(AVG(s.hardship_index), 1) AS avg_hardship_index,
    ROUND(AVG(s.poverty_rate), 1) AS avg_poverty_rate,
    ROUND(AVG(s.unemployment_rate), 1) AS avg_unemployment_rate,
    ROUND(AVG(s.no_high_school_diploma_rate), 1) AS avg_no_high_school_diploma_rate,
    ROUND(AVG(s.crowded_housing_rate), 1) AS avg_crowded_housing_rate,
    ROUND(AVG(s.dependency_rate), 1) AS avg_dependency_rate,
    ROUND(AVG(s.per_capita_income), 0) AS avg_per_capita_income

FROM crime_area_summary c
JOIN socio s 
    ON c.community_area = s.community_area

GROUP BY hardship_group

ORDER BY CASE
    WHEN hardship_group = 'Low Hardship' THEN 1
    WHEN hardship_group = 'Mid Hardship' THEN 2
    WHEN hardship_group = 'High Hardship' THEN 3
END;
''')

,hardship_group,num_communities,avg_total_crimes,avg_violent_crimes,avg_violent_share_pct,avg_domestic_share_pct,avg_arrest_rate_pct,avg_hardship_index,avg_poverty_rate,avg_unemployment_rate,avg_no_high_school_diploma_rate,avg_crowded_housing_rate,avg_dependency_rate,avg_per_capita_income
0,Low Hardship,38,13625.0,3538.0,26.1,15.2,10.1,24.5,13.8,10.2,12.1,2.9,31.7,35914.0
1,Mid Hardship,19,17259.0,6041.0,32.7,23.2,12.0,61.1,22.6,17.7,25.3,5.8,38.5,17905.0
2,High Hardship,20,16961.0,6321.0,36.9,24.4,15.1,86.0,36.1,23.0,31.3,8.0,40.9,13173.0


This query groups Chicago community areas into low-, mid-, and high-hardship categories, then compares both crime composition and broader socioeconomic conditions across the groups. It supports the Python finding that hardship is more closely related to crime composition than to total reported crime volume alone.

The output shows that average total crime does not increase mechanically with hardship. Low-hardship communities average 13,625 reported crimes, mid-hardship communities average 17,259, and high-hardship communities average 16,961. However, crime composition changes clearly across hardship groups. Average violent-crime share rises from 26.1% in low-hardship communities to 32.7% in mid-hardship communities and 36.9% in high-hardship communities. Domestic-related share also increases from 15.2% to 23.2% and then 24.4%. Arrest rate follows the same upward pattern, rising from 10.1% to 12.0% and then 15.1%.

The broader socioeconomic indicators show a consistent disadvantage gradient. Average poverty rate rises from 13.8% in low-hardship communities to 22.6% in mid-hardship communities and 36.1% in high-hardship communities. Average unemployment rises from 10.2% to 17.7% and then 23.0%. The no-high-school-diploma rate increases from 12.1% to 25.3% and then 31.3%, while crowded housing rises from 2.9% to 5.8% and then 8.0%. Average per capita income moves in the opposite direction, falling from about $35,914 in low-hardship communities to $17,905 in mid-hardship communities and $13,173 in high-hardship communities.

This result shows that the hardship index is not acting as an isolated variable. It reflects a broader pattern of socioeconomic disadvantage across poverty, unemployment, education, housing conditions, dependency, and income. SQL therefore complements the Python analysis by confirming the hardship-crime composition pattern through grouped aggregation and by extending the interpretation to a wider set of socioeconomic indicators.

<h2>Part C: SQL Evidence that Crime Volume and Severity Differ (Query 7–10)</h2>

<h3> Query 7: High-volume but lower-violent-share communities

In [37]:
run_sql("""
SELECT
    s.community_area_name,
    c.total_crimes,
    ROUND(c.violent_crime_share, 4) AS violent_crime_share,
    ROUND(s.hardship_index, 1) AS hardship_index,
    ROUND(s.poverty_rate, 1) AS poverty_rate,
    ROUND(s.per_capita_income, 0) AS per_capita_income
FROM crime_area_summary c
JOIN socio s ON c.community_area = s.community_area
WHERE c.total_crimes > (SELECT AVG(total_crimes) FROM crime_area_summary)
  AND c.violent_crime_share < (SELECT AVG(violent_crime_share) FROM crime_area_summary)
ORDER BY c.total_crimes DESC
LIMIT 10;
""")

,community_area_name,total_crimes,violent_crime_share,hardship_index,poverty_rate,per_capita_income
0,Near North Side,49009,0.2440,1.0,12.9,88669.0
1,Near West Side,44421,0.2705,15.0,20.6,44689.0
2,Loop,36932,0.2324,3.0,14.7,65526.0
3,West Town,34249,0.2330,10.0,14.7,43198.0
4,Lake View,28361,0.2203,5.0,11.4,60058.0
5,Logan Square,22217,0.2437,23.0,16.8,31908.0
6,Uptown,19193,0.3055,20.0,24.0,35787.0
7,West Ridge,17912,0.2824,46.0,17.2,23040.0
8,Lincoln Park,17696,0.1686,2.0,12.3,71551.0


This query identifies communities with above-average total reported crime but below-average violent-crime share. It supports the Python finding that total crime volume and violent-crime share measure different dimensions of neighborhood safety.

The output highlights communities such as Near North Side, Near West Side, the Loop, West Town, Lake View, Logan Square, Uptown, West Ridge, and Lincoln Park. These communities have relatively high reported crime volume, but their violent-crime shares are below the citywide community average. Many of these areas also have low hardship indices and relatively high per capita income. For example, Near North Side has 49,009 reported crimes but a violent-crime share of only 24.4% and a hardship index of 1.0. Lincoln Park has a violent-crime share of only 16.9% and a hardship index of 2.0.

This query adds depth to the Python analysis by identifying high-volume but lower-severity communities. These areas may have high reported crime counts because of commercial activity, visitor exposure, nightlife, retail density, transportation activity, or reporting density, rather than because violent incidents make up a large share of crime. This supports the project’s main argument that total reported crime count alone is not enough to describe neighborhood safety.

<h3> Query 8: Low-volume but higher-violent-share communities

In [39]:
run_sql("""
SELECT
    s.community_area_name,
    c.total_crimes,
    c.violent_crimes,
    ROUND(c.violent_crime_share, 4) AS violent_crime_share,
    ROUND(s.hardship_index, 1) AS hardship_index,
    ROUND(s.poverty_rate, 1) AS poverty_rate,
    ROUND(s.per_capita_income, 0) AS per_capita_income
FROM crime_area_summary c
JOIN socio s ON c.community_area = s.community_area
WHERE c.total_crimes < (SELECT AVG(total_crimes) FROM crime_area_summary)
  AND c.violent_crime_share > (SELECT AVG(violent_crime_share) FROM crime_area_summary)
ORDER BY c.violent_crime_share DESC
LIMIT 10;
""")

,community_area_name,total_crimes,violent_crimes,violent_crime_share,hardship_index,poverty_rate,per_capita_income
0,Riverdale,5948,2510,0.4220,98.0,56.5,8201.0
1,Fuller Park,3322,1281,0.3856,97.0,51.2,10432.0
2,Washington Park,11239,4270,0.3799,88.0,42.1,13785.0
3,Brighton Park,10278,3848,0.3744,84.0,23.6,13089.0
4,Gage Park,10291,3667,0.3563,93.0,23.4,12171.0
5,Armour Square,5185,1794,0.3460,82.0,40.1,16148.0
6,East Side,5919,1995,0.3371,64.0,19.2,17104.0
7,South Deering,8852,2925,0.3304,65.0,29.2,14685.0
8,Hermosa,6885,2273,0.3301,71.0,20.5,15089.0
9,Lower West Side,13397,4422,0.3301,76.0,25.8,16444.0


This query identifies communities with below-average total reported crime but above-average violent-crime share. It complements Query 7 by showing the opposite pattern: some communities may not appear at the top of total-crime rankings, but violent incidents make up a relatively large share of their reported crime.

The output highlights communities such as Riverdale, Fuller Park, Washington Park, Brighton Park, and Gage Park. These areas have lower total crime volume than the citywide community average, but their violent-crime shares are relatively high. Many of them also have high hardship indices, high poverty rates, and low per capita income. For example, Riverdale has a violent-crime share of 42.2%, a hardship index of 98.0, and a poverty rate of 56.5%. Fuller Park has a violent-crime share of 38.6%, a hardship index of 97.0, and a poverty rate of 51.2%.

This query adds depth to the Python analysis by identifying hidden severity communities. These areas could be overlooked if the analysis only ranks neighborhoods by total reported crime count. For public safety and community development purposes, this reinforces the need to evaluate both crime volume and crime composition.

<h3> Query 9: Violent-crime share ranking with SQL Window Function 

In [30]:
run_sql('''
SELECT
    ROW_NUMBER() OVER (ORDER BY c.violent_crime_share DESC) AS violent_crime_rank,
    s.community_area_name,
    c.total_crimes,
    c.violent_crimes,
    ROUND(c.violent_crime_share, 4) AS violent_crime_share,
    ROUND(s.hardship_index, 1) AS hardship_index
FROM crime_area_summary c
JOIN socio s ON c.community_area = s.community_area
ORDER BY c.violent_crime_share DESC;
''')

,violent_crime_rank,community_area_name,total_crimes,violent_crimes,violent_crime_share,hardship_index
0,1,Riverdale,5948,2510,0.4220,98.0
1,2,Englewood,25044,10127,0.4044,94.0
2,3,New City,19511,7687,0.3940,91.0
3,4,South Lawndale,19502,7669,0.3932,96.0
4,5,North Lawndale,34464,13326,0.3867,87.0
...,...,...,...,...,...,...
72,73,Edison Park,1396,286,0.2049,8.0
73,74,O'Hare,8063,1605,0.1991,24.0
74,75,North Center,6452,1228,0.1903,6.0
75,76,Forest Glen,2739,487,0.1778,11.0


This query ranks all 77 Chicago community areas by violent-crime share using a SQL window function, `ROW_NUMBER()`. Unlike total-crime rankings, this ranking focuses on crime composition rather than the absolute number of reported incidents. The purpose is to create a severity-based ranking that can be compared with the earlier volume-based analysis.

The output shows that the highest violent-crime-share communities include Riverdale, Englewood, New City, South Lawndale, and North Lawndale. These communities have violent-crime shares around 38.7% to 42.2%, and they also have high hardship indices. For example, Riverdale ranks first with a violent-crime share of 42.2% and a hardship index of 98.0, while Englewood ranks second with a violent-crime share of 40.4% and a hardship index of 94.0.

At the bottom of the ranking, communities such as Edison Park, O’Hare, North Center, and Forest Glen have much lower violent-crime shares and generally lower hardship indices. This ranking reinforces the main finding that violent-crime share provides a different view of neighborhood risk than total crime volume. SQL adds value here by using a window function to generate a full severity ranking across all community areas.

<h3> Query 10: Extreme Community Comparison — Highest vs Lowest Crime Volume Groups

In [23]:
run_sql('''
WITH ranked_communities AS (
    SELECT
        s.community_area_name,
        c.total_crimes,
        c.violent_crime_share,
        c.domestic_share,
        c.arrest_rate,
        s.hardship_index,
        s.poverty_rate,
        s.unemployment_rate,
        s.no_high_school_diploma_rate,
        s.crowded_housing_rate,
        s.dependency_rate,
        s.per_capita_income,
        ROW_NUMBER() OVER (ORDER BY c.total_crimes DESC) AS high_crime_rank,
        ROW_NUMBER() OVER (ORDER BY c.total_crimes ASC) AS low_crime_rank
    FROM crime_area_summary c
    JOIN socio s
        ON c.community_area = s.community_area
),

extreme_groups AS (
    SELECT
        CASE
            WHEN high_crime_rank <= 10 THEN 'Top 10 highest-crime communities'
            WHEN low_crime_rank <= 10 THEN 'Bottom 10 lowest-crime communities'
        END AS crime_volume_group,
        community_area_name,
        total_crimes,
        violent_crime_share,
        domestic_share,
        arrest_rate,
        hardship_index,
        poverty_rate,
        unemployment_rate,
        no_high_school_diploma_rate,
        crowded_housing_rate,
        dependency_rate,
        per_capita_income
    FROM ranked_communities
    WHERE high_crime_rank <= 10 OR low_crime_rank <= 10
)

SELECT
    crime_volume_group,
    COUNT(*) AS num_communities,
    ROUND(AVG(total_crimes), 0) AS avg_total_crimes,
    ROUND(AVG(violent_crime_share) * 100, 1) AS avg_violent_share_pct,
    ROUND(AVG(domestic_share) * 100, 1) AS avg_domestic_share_pct,
    ROUND(AVG(arrest_rate) * 100, 1) AS avg_arrest_rate_pct,
    ROUND(AVG(hardship_index), 1) AS avg_hardship_index,
    ROUND(AVG(poverty_rate), 1) AS avg_poverty_rate,
    ROUND(AVG(unemployment_rate), 1) AS avg_unemployment_rate,
    ROUND(AVG(no_high_school_diploma_rate), 1) AS avg_no_high_school_diploma_rate,
    ROUND(AVG(crowded_housing_rate), 1) AS avg_crowded_housing_rate,
    ROUND(AVG(dependency_rate), 1) AS avg_dependency_rate,
    ROUND(AVG(per_capita_income), 0) AS avg_per_capita_income,
    GROUP_CONCAT(community_area_name, ', ') AS example_communities
FROM extreme_groups
GROUP BY crime_volume_group
ORDER BY avg_total_crimes DESC;
''')

,crime_volume_group,num_communities,avg_total_crimes,avg_violent_share_pct,avg_domestic_share_pct,avg_arrest_rate_pct,avg_hardship_index,avg_poverty_rate,avg_unemployment_rate,avg_no_high_school_diploma_rate,avg_crowded_housing_rate,avg_dependency_rate,avg_per_capita_income,example_communities
0,Top 10 highest-crime communities,10,39864.0,32.0,19.1,15.2,46.9,25.7,16.2,16.4,4.8,31.7,33607.0,"Austin, Near North Side, Near West Side, South..."
1,Bottom 10 lowest-crime communities,10,3086.0,28.2,20.4,9.6,51.3,20.5,15.7,19.4,4.4,39.5,23910.0,"West Elsdon, McKinley Park, Oakland, Fuller Pa..."


This query compares the two extremes of the crime-volume distribution: the 10 communities with the highest total reported crime and the 10 communities with the lowest total reported crime. Instead of only listing the communities separately, SQL aggregates each extreme group and compares their average crime composition and socioeconomic characteristics.

The output shows a very large difference in reported crime volume. The top 10 highest-crime communities average 39,864 crimes, while the bottom 10 lowest-crime communities average only 3,086 crimes. This is roughly a 12.9x difference in total reported crime volume. However, the socioeconomic comparison is not one-dimensional. The top-crime group has an average hardship index of 46.9, while the bottom-crime group has a slightly higher average hardship index of 51.3. The bottom-crime group also has a higher average no-high-school-diploma rate, 19.4% compared with 16.4% for the top-crime group.

This result is important because it shows that low reported crime volume does not automatically mean low socioeconomic hardship, and high reported crime volume does not always mean the community is more disadvantaged. Some high-crime communities may reflect high activity density, commercial land use, visitor exposure, and reporting density, while some low-crime communities may still face socioeconomic stress but have lower reported incident volume.

This final comparison reinforces the project’s main argument: total reported crime count alone is not enough to describe neighborhood safety. Crime volume, crime composition, and socioeconomic context need to be interpreted together.

<h2> Part D: Further Investigation of Broader Socioeconomic Indicators: </h2>Part D extends the SQL analysis beyond the main Python findings by examining additional socioeconomic indicators that were not emphasized in the Python notebook. These queries focus on unemployment, age dependency, and the combined unemployment-dependency profile. The goal is not to make causal claims, but to explore whether broader socioeconomic conditions are associated with different crime profiles.

<h3> Query 11: Crime Profile by Unemployment Group

In [26]:
run_sql('''
SELECT
    CASE
        WHEN s.unemployment_rate < 10 THEN 'Low Unemployment'
        WHEN s.unemployment_rate >= 10 AND s.unemployment_rate < 20 THEN 'Mid Unemployment'
        ELSE 'High Unemployment'
    END AS unemployment_group,

    COUNT(*) AS num_communities,

    -- Crime profile
    ROUND(AVG(c.total_crimes), 0) AS avg_total_crimes,
    ROUND(AVG(c.violent_crime_share) * 100, 1) AS avg_violent_share_pct,
    ROUND(AVG(c.domestic_share) * 100, 1) AS avg_domestic_share_pct,
    ROUND(AVG(c.arrest_rate) * 100, 1) AS avg_arrest_rate_pct,

    -- Socioeconomic context
    ROUND(AVG(s.unemployment_rate), 1) AS avg_unemployment_rate,
    ROUND(AVG(s.poverty_rate), 1) AS avg_poverty_rate,
    ROUND(AVG(s.no_high_school_diploma_rate), 1) AS avg_no_high_school_diploma_rate,
    ROUND(AVG(s.per_capita_income), 0) AS avg_per_capita_income,
    ROUND(AVG(s.hardship_index), 1) AS avg_hardship_index

FROM crime_area_summary c
JOIN socio s
    ON c.community_area = s.community_area

GROUP BY unemployment_group

ORDER BY CASE
    WHEN unemployment_group = 'Low Unemployment' THEN 1
    WHEN unemployment_group = 'Mid Unemployment' THEN 2
    WHEN unemployment_group = 'High Unemployment' THEN 3
END;
''')

,unemployment_group,num_communities,avg_total_crimes,avg_violent_share_pct,avg_domestic_share_pct,avg_arrest_rate_pct,avg_unemployment_rate,avg_poverty_rate,avg_no_high_school_diploma_rate,avg_per_capita_income,avg_hardship_index
0,Low Unemployment,25,14177.0,24.7,13.2,10.0,7.7,13.0,11.3,39955.0,20.3
1,Mid Unemployment,31,12591.0,31.5,20.4,11.8,14.8,21.7,27.5,19938.0,58.6
2,High Unemployment,21,20959.0,36.2,25.9,14.2,25.3,32.2,20.6,16733.0,70.8


This query extends the SQL analysis beyond the main Python findings by examining unemployment as an additional socioeconomic indicator. Communities are grouped into low-, mid-, and high-unemployment categories, and each group is compared by crime volume, violent-crime share, domestic-related share, arrest rate, and broader socioeconomic context.

The output shows that total crime volume does not increase perfectly across unemployment groups. Low-unemployment communities average 14,177 reported crimes, mid-unemployment communities average 12,591, and high-unemployment communities average 20,959. However, crime composition changes more clearly. Average violent-crime share rises from 24.7% in low-unemployment communities to 31.5% in mid-unemployment communities and 36.2% in high-unemployment communities. Domestic-related share also increases from 13.2% to 20.4% and then 25.9%. Arrest rate follows the same upward pattern, rising from 10.0% to 11.8% and then 14.2%.

The socioeconomic context also shifts across unemployment groups. Average poverty rises from 13.0% in low-unemployment communities to 21.7% in mid-unemployment communities and 32.2% in high-unemployment communities. Average per capita income falls from about $39,955 to $19,938 and then $16,733. Hardship index also increases from 20.3 to 58.6 and then 70.8. These patterns suggest that unemployment is part of a broader socioeconomic disadvantage profile rather than an isolated variable.

This query should be interpreted as descriptive association, not causation. It does not prove that unemployment causes crime. Instead, it shows that communities with higher unemployment tend to have a more severe crime composition, especially higher violent-crime and domestic-related shares. This broadens the SQL analysis beyond hardship alone.

<h3> Query 12: Crime Profile by Age Dependency Group

In [28]:
run_sql('''
SELECT
    CASE
        WHEN s.dependency_rate < 30 THEN 'Low Dependency'
        WHEN s.dependency_rate >= 30 AND s.dependency_rate < 40 THEN 'Mid Dependency'
        ELSE 'High Dependency'
    END AS dependency_group,

    COUNT(*) AS num_communities,

    -- Crime profile
    ROUND(AVG(c.total_crimes), 0) AS avg_total_crimes,
    ROUND(AVG(c.violent_crime_share) * 100, 1) AS avg_violent_share_pct,
    ROUND(AVG(c.domestic_share) * 100, 1) AS avg_domestic_share_pct,
    ROUND(AVG(c.arrest_rate) * 100, 1) AS avg_arrest_rate_pct,

    -- Socioeconomic context
    ROUND(AVG(s.dependency_rate), 1) AS avg_dependency_rate,
    ROUND(AVG(s.unemployment_rate), 1) AS avg_unemployment_rate,
    ROUND(AVG(s.poverty_rate), 1) AS avg_poverty_rate,
    ROUND(AVG(s.no_high_school_diploma_rate), 1) AS avg_no_high_school_diploma_rate,
    ROUND(AVG(s.crowded_housing_rate), 1) AS avg_crowded_housing_rate,
    ROUND(AVG(s.per_capita_income), 0) AS avg_per_capita_income,
    ROUND(AVG(s.hardship_index), 1) AS avg_hardship_index

FROM crime_area_summary c
JOIN socio s
    ON c.community_area = s.community_area

GROUP BY dependency_group

ORDER BY CASE
    WHEN dependency_group = 'Low Dependency' THEN 1
    WHEN dependency_group = 'Mid Dependency' THEN 2
    WHEN dependency_group = 'High Dependency' THEN 3
END;
''')

,dependency_group,num_communities,avg_total_crimes,avg_violent_share_pct,avg_domestic_share_pct,avg_arrest_rate_pct,avg_dependency_rate,avg_unemployment_rate,avg_poverty_rate,avg_no_high_school_diploma_rate,avg_crowded_housing_rate,avg_per_capita_income,avg_hardship_index
0,Low Dependency,14,23179.0,24.9,9.9,10.3,22.7,7.3,15.7,8.5,2.6,49392.0,12.9
1,Mid Dependency,39,12253.0,30.3,19.4,11.6,36.4,14.4,19.1,24.7,6.2,21495.0,52.1
2,High Dependency,24,15939.0,34.3,25.5,13.2,42.4,21.6,29.6,20.2,4.2,18273.0,66.7


This query extends the SQL analysis by examining age dependency as an additional socioeconomic and demographic indicator. Dependency rate measures the share of residents who are under 18 or over 64, so it captures part of a community’s age structure and potential social vulnerability. The query groups communities into low-, mid-, and high-dependency categories, then compares crime volume, crime composition, arrest rate, and broader socioeconomic context across those groups.

The output shows that total crime volume does not move in a simple linear pattern across dependency groups. Low-dependency communities average 23,179 reported crimes, mid-dependency communities average 12,253, and high-dependency communities average 15,939. However, crime composition shows a clearer upward pattern. Average violent-crime share increases from 24.9% in low-dependency communities to 30.3% in mid-dependency communities and 34.3% in high-dependency communities. Domestic-related share rises even more sharply, from 9.9% to 19.4% and then 25.5%. Arrest rate also increases from 10.3% to 11.6% and then 13.2%.

The socioeconomic context also changes across dependency groups. Average unemployment rises from 7.3% in low-dependency communities to 14.4% in mid-dependency communities and 21.6% in high-dependency communities. Poverty increases from 15.7% to 19.1% and then 29.6%, while average per capita income falls from about $49,392 to $21,495 and then $18,273. The hardship index also rises from 12.9 to 52.1 and then 66.7.

This result suggests that age dependency is associated with broader neighborhood vulnerability and a more severe crime composition, especially higher violent-crime and domestic-related shares. However, this should be interpreted as a descriptive relationship rather than a causal effect. The query broadens the analysis by showing that community risk profiles may vary with demographic structure as well as income, poverty, unemployment, and hardship.

<h3> Query 13 Combined Unemployment and Dependency Risk Profile 

In [30]:
run_sql('''
WITH socioeconomic_flags AS (
    SELECT
        c.community_area,
        s.community_area_name,
        c.total_crimes,
        c.violent_crime_share,
        c.domestic_share,
        c.arrest_rate,

        s.unemployment_rate,
        s.dependency_rate,
        s.poverty_rate,
        s.no_high_school_diploma_rate,
        s.crowded_housing_rate,
        s.per_capita_income,
        s.hardship_index,

        CASE
            WHEN s.unemployment_rate >= (SELECT AVG(unemployment_rate) FROM socio)
             AND s.dependency_rate >= (SELECT AVG(dependency_rate) FROM socio)
                THEN 'High unemployment / High dependency'
            WHEN s.unemployment_rate >= (SELECT AVG(unemployment_rate) FROM socio)
             AND s.dependency_rate < (SELECT AVG(dependency_rate) FROM socio)
                THEN 'High unemployment / Lower dependency'
            WHEN s.unemployment_rate < (SELECT AVG(unemployment_rate) FROM socio)
             AND s.dependency_rate >= (SELECT AVG(dependency_rate) FROM socio)
                THEN 'Lower unemployment / High dependency'
            ELSE 'Lower unemployment / Lower dependency'
        END AS unemployment_dependency_type

    FROM crime_area_summary c
    JOIN socio s
        ON c.community_area = s.community_area
)

SELECT
    unemployment_dependency_type,
    COUNT(*) AS num_communities,

    -- Crime profile
    ROUND(AVG(total_crimes), 0) AS avg_total_crimes,
    ROUND(AVG(violent_crime_share) * 100, 1) AS avg_violent_share_pct,
    ROUND(AVG(domestic_share) * 100, 1) AS avg_domestic_share_pct,
    ROUND(AVG(arrest_rate) * 100, 1) AS avg_arrest_rate_pct,

    -- Socioeconomic context
    ROUND(AVG(unemployment_rate), 1) AS avg_unemployment_rate,
    ROUND(AVG(dependency_rate), 1) AS avg_dependency_rate,
    ROUND(AVG(poverty_rate), 1) AS avg_poverty_rate,
    ROUND(AVG(no_high_school_diploma_rate), 1) AS avg_no_high_school_diploma_rate,
    ROUND(AVG(crowded_housing_rate), 1) AS avg_crowded_housing_rate,
    ROUND(AVG(per_capita_income), 0) AS avg_per_capita_income,
    ROUND(AVG(hardship_index), 1) AS avg_hardship_index,

    GROUP_CONCAT(community_area_name, ', ') AS example_communities

FROM socioeconomic_flags
GROUP BY unemployment_dependency_type
ORDER BY avg_violent_share_pct DESC;
''')

,unemployment_dependency_type,num_communities,avg_total_crimes,avg_violent_share_pct,avg_domestic_share_pct,avg_arrest_rate_pct,avg_unemployment_rate,avg_dependency_rate,avg_poverty_rate,avg_no_high_school_diploma_rate,avg_crowded_housing_rate,avg_per_capita_income,avg_hardship_index,example_communities
0,High unemployment / High dependency,31,17727.0,35.3,24.5,14.0,22.8,41.0,31.0,24.3,5.7,15888.0,73.3,"Humboldt park, Austin, West Garfield Park, Eas..."
1,High unemployment / Lower dependency,5,19153.0,34.4,21.0,11.2,17.1,33.6,27.8,27.0,6.4,21189.0,60.0,"South Lawndale, Lower West Side, Douglas, Kenw..."
2,Lower unemployment / High dependency,17,7760.0,27.8,20.0,9.6,10.9,39.1,13.1,21.1,4.9,25000.0,41.5,"West Ridge, Norwood Park, Forest Glen, North P..."
3,Lower unemployment / Lower dependency,24,16987.0,25.6,12.6,10.8,8.6,27.0,14.8,13.4,3.7,39371.0,22.3,"Rogers Park, Uptown, Lincoln Square, North Cen..."


This query extends the further investigation section by combining unemployment and age dependency into a two-factor socioeconomic profile. Instead of examining each variable separately, it classifies communities based on whether they are above or below the citywide average for both unemployment rate and dependency rate. This helps test whether overlapping labor-market stress and demographic vulnerability are associated with different crime profiles.

The output shows that the high unemployment / high dependency group contains 31 communities and has the highest violent-crime share at 35.3%, the highest domestic-related share at 24.5%, and the highest arrest rate at 14.0%. This group also has the highest average hardship index, 73.3, along with high poverty at 31.0%, low per capita income at about $15,888, and elevated educational disadvantage. These patterns suggest that communities facing both high unemployment and high dependency tend to have a more severe crime composition.

The high unemployment / lower dependency group also has a high violent-crime share of 34.4% and domestic-related share of 21.0%, but it includes only 5 communities. The lower unemployment / high dependency group has lower average total crime volume and a lower violent-crime share of 27.8%, while the lower unemployment / lower dependency group has the lowest violent-crime share at 25.6% and the lowest domestic-related share at 12.6%.

This result supports the broader conclusion that neighborhood risk is multidimensional. Unemployment and dependency do not explain crime patterns by themselves, and this query should not be interpreted causally. However, the grouped comparison suggests that communities with overlapping socioeconomic pressures tend to show higher violent-crime and domestic-related shares. This strengthens the SQL extension by moving beyond hardship alone and examining how multiple socioeconomic indicators interact in community-level risk profiles.

## SQL Summary

The SQL analysis serves two main purposes. First, it supports and deepens the main findings from the Python notebook. The early validation queries confirm that the cleaned incident-level crime data were successfully loaded into SQLite, that all three analysis tables cover Chicago’s 77 community areas, and that crime measures can be joined with socioeconomic indicators at the community-area level. After this setup, the SQL queries use joins, grouped aggregations, CASE statements, CTEs, filters, subqueries, and a window function to examine the same core ideas from the Python EDA in more detail.

The SQL results support the Python finding that reported crime is concentrated across communities, but they also show that high total crime volume does not represent one single type of neighborhood risk. Among high-crime communities, some areas have high hardship, high poverty, high unemployment, lower educational attainment, and higher violent-crime and domestic-related shares. Other high-crime communities have much lower hardship and lower violent-crime shares, suggesting that their reported crime volume may partly reflect commercial activity, visitor exposure, land use, nightlife, retail density, or reporting density. This adds depth to the Python analysis by showing that high-volume communities are not socioeconomically or compositionally identical.

The SQL analysis also reinforces the project’s main argument that crime composition matters. Queries comparing high-volume / lower-violent-share communities with low-volume / higher-violent-share communities show that total crime count and violent-crime share capture different dimensions of neighborhood safety. Some communities, such as Near North Side, the Loop, West Town, Lake View, and Lincoln Park, have high reported crime volume but relatively lower violent-crime shares. In contrast, communities such as Riverdale, Fuller Park, Washington Park, and Gage Park may have lower total crime volume but higher violent-crime shares. These communities could be overlooked if the analysis only ranked neighborhoods by total reported incidents.

Finally, the SQL analysis extends the Python notebook by investigating broader socioeconomic indicators beyond hardship, poverty, and income. Additional queries examine unemployment, age dependency, education level, crowded housing, and combined unemployment-dependency profiles. These exploratory results suggest that higher unemployment and higher dependency are associated with higher violent-crime and domestic-related shares, although the results should be interpreted descriptively rather than causally. Overall, the SQL analysis strengthens the project’s conclusion that neighborhood safety should be evaluated using crime volume, crime composition, and broader socioeconomic context together. Total reported crime count alone is not enough.